#For week 3 we are doing some data Preprocessing, utilizing the alert in the groupchat we will not using lisitng/sale dependent variables (`ListPrice`, `OriginalListPrice`,`DaysOnMarket`,`PurchaseContractDate`, and `ListingContractDate`) for model inputs. 

The final model should be able to predict closeprice for any single-family address in california (this should include homes that are not listed!)

We should filter all the data to only residential and singlefamilyresidence and we need to make sure to restrict the model population to California (Just is case any data points have a location outsdie CA)

In [11]:
import pandas as pd
import numpy as np
import os

os.chdir(r"C:\Users\sejal\IDX\IDX-Exchange-Data-Science")
# get data
#using data from 05/2025 to 04/2026
files = [
    "Data/CRMLSSold202505.csv",
    "Data/CRMLSSold202506.csv",
    "Data/CRMLSSold202507.csv",
    "Data/CRMLSSold202508.csv",
    "Data/CRMLSSold202509.csv",
    "Data/CRMLSSold202510.csv",
    "Data/CRMLSSold202511.csv",
    "Data/CRMLSSold202512.csv",
    "Data/CRMLSSold202601.csv",
    "Data/CRMLSSold202602.csv",
    "Data/CRMLSSold202603.csv",
    "Data/CRMLSSold202604.csv"
]

#load
all_data = []

for file in files:
    temp = pd.read_csv(file, low_memory=False)
    all_data.append(temp)

#combine
df = pd.concat(all_data, ignore_index=True)
#shape

print(df.shape)
#we should use errors='coerce' just in case we do see bad data, we don't want everything to crash too.
df["CloseDate"] = pd.to_datetime(df["CloseDate"], errors="coerce")
print(df["CloseDate"].min())
print(df["CloseDate"].max())

#lets filter data only to include residential and singlefamilyresidential homes
sfh_df = df[
    (df["PropertyType"] == "Residential") &
    (df["PropertySubType"] == "SingleFamilyResidence")
].copy()

#quick check on amount of remaining data print("Original shape:", df.shape)
print("Single-family residential shape:", sfh_df.shape)

(261087, 80)
2025-05-01 00:00:00
2026-04-30 00:00:00
Single-family residential shape: (131226, 80)


In [12]:
# keep only properties located in California
print(sfh_df["StateOrProvince"].value_counts(dropna=False))
sfh_df = sfh_df[
    sfh_df["StateOrProvince"] == "CA"
].copy()

print("California single-family residential shape:", sfh_df.shape)
print("\n")
#lets see if removing other states was sucessful.
print(sfh_df["StateOrProvince"].value_counts(dropna=False))


StateOrProvince
CA    131220
AZ         3
CO         1
FL         1
OS         1
Name: count, dtype: int64
California single-family residential shape: (131220, 80)


StateOrProvince
CA    131220
Name: count, dtype: int64


In [13]:
#lets prohibit features that should not be used in the model
prohibited_features = [
    "ListPrice",
    "OriginalListPrice",
    "DaysOnMarket",
    "PurchaseContractDate",
    "ListingContractDate"
]

#define target
target = "ClosePrice"

#before drop lets do quick check
existing_prohibited = [
    col for col in prohibited_features
    if col in sfh_df.columns
]

print("Prohibited columns found:")
print(existing_prohibited)

for col in sfh_df.columns:
    if (
        "List" in col or
        "Market" in col or
        "Contract" in col
    ):
        print(col)

Prohibited columns found:
['ListPrice', 'OriginalListPrice', 'DaysOnMarket', 'PurchaseContractDate', 'ListingContractDate']
ListAgentAOR
OriginalListPrice
ListingKey
ListAgentEmail
ListAgentFirstName
ListAgentLastName
ListPrice
DaysOnMarket
ListOfficeName
CoListOfficeName
ListAgentFullName
CoListAgentFirstName
CoListAgentLastName
ListingKeyNumeric
ListingId
ContractStatusChangeDate
PurchaseContractDate
ListingContractDate


Using what we know about the missing value analysis completed in the EDA, lets focus on cleaning up the data and making decisions for final modeling dataset

In [14]:
#features we want
candidate_features = [
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "YearBuilt",
    "PostalCode",
    "City"
]

#do quick missingness sanity check for candidate features and target
check_cols = candidate_features + ["ClosePrice", "CloseDate"]

sfh_df[check_cols].isna().sum().sort_values(ascending=False)

LotSizeSquareFeet        2262
YearBuilt                  93
LivingArea                 73
City                       40
BathroomsTotalInteger      13
PostalCode                  1
BedroomsTotal               0
ClosePrice                  0
CloseDate                   0
dtype: int64

We know we have around 131k rows of data so these missingness shown here is tiny, so we shouldn't drop themn, instead lets find a way to handle them. For this with 0 missing values we will not need to do anything.

For the rest we may drop, impute using median or add missing value flags. But we cannot calculate these medians yet, our test set is the most recent month so we want to split by time first and medians should ONLY be learned fron TRAINING data!!!!!

In [15]:
#month variable creation

sfh_df["CloseMonth"] = sfh_df["CloseDate"].dt.to_period("M")

print(sfh_df["CloseMonth"].value_counts().sort_index())

#find newest month
test_month = sfh_df["CloseMonth"].max()

print("Test month:", test_month)

CloseMonth
2025-05    11776
2025-06    11700
2025-07    12114
2025-08    11452
2025-09    11456
2025-10    12028
2025-11     9739
2025-12    10455
2026-01     7490
2026-02     8959
2026-03    11562
2026-04    12489
Freq: M, Name: count, dtype: int64
Test month: 2026-04


In [16]:
#lets seperate April 2026 data as its the most recent.

test_df = sfh_df[
    sfh_df["CloseMonth"] == test_month
].copy()

historical_df = sfh_df[
    sfh_df["CloseMonth"] < test_month
].copy()

print("Historical data shape:", historical_df.shape)
print("Test data shape:", test_df.shape)

# missing-value statistics will be calculated after selecting training window X

Historical data shape: (118731, 81)
Test data shape: (12489, 81)


So far, we have reserved the most recent month, April 2026, as the final test set.
The remaining historical data will be used to experiment with different training-window lengths X.

Now we will do the missing value data preprocessing after a specific training window is selected.

Numeric features such as `LivingArea`, `LotSizeSquareFeet`, `YearBuilt`, and
`BathroomsTotalInteger` will be imputed using statistics learned only from the
training data to avoid test-set leakage.

Categorical features such as `City` and `PostalCode` will be assigned an
`Unknown` category when missing.

Because `LotSizeSquareFeet` has noticeably more missing values than the other
candidate features, a binary indicator will also record whether lot size was
originally missing.


In [17]:
numeric_impute_cols = [
    "LotSizeSquareFeet",
    "YearBuilt",
    "LivingArea",
    "BathroomsTotalInteger"
]

categorical_fill_cols = [
    "City",
    "PostalCode"
]

historical_df["LotSizeMissing"] = (
    historical_df["LotSizeSquareFeet"].isna().astype(int)
)

test_df["LotSizeMissing"] = (
    test_df["LotSizeSquareFeet"].isna().astype(int)
)

print("Historical lot-size missing flag:")
print(historical_df["LotSizeMissing"].value_counts())

print("\nTest lot-size missing flag:")
print(test_df["LotSizeMissing"].value_counts())

Historical lot-size missing flag:
LotSizeMissing
0    116711
1      2020
Name: count, dtype: int64

Test lot-size missing flag:
LotSizeMissing
0    12247
1      242
Name: count, dtype: int64


Now we will creat e avlaidation set for the training window selection, we already know our final set has data from April 2026. Lets use the month before, so march 2026 as the validation month, different training windows will be trained using months befre march 2026 and will be tested on how well they predict march.

In [18]:
#we will use validation set to act as a practice test before testing with april data, lets use the month before as validation set
validation_month = test_month - 1

print("Validation month:", validation_month)
print("Final test month:", test_month)

Validation month: 2026-03
Final test month: 2026-04


In [19]:
# March 2026 will be used to compare different training-window lengths
validation_df = historical_df[
    historical_df["CloseMonth"] == validation_month
].copy()

# only months before March can be used to train during window comparison
pre_validation_df = historical_df[
    historical_df["CloseMonth"] < validation_month
].copy()

print("Validation shape:", validation_df.shape)
print("Data available before validation:", pre_validation_df.shape)

Validation shape: (11562, 82)
Data available before validation: (107169, 82)


In [20]:
#candidate training-window lengths
validation_windows = [3, 6, 9, 10]

#lets use a helper function that gives the correct rows for chosen X

def get_training_window(df, reference_month, months):
    start_month = reference_month - months

    train_df = df[
        (df["CloseMonth"] >= start_month) &
        (df["CloseMonth"] < reference_month)
    ].copy()

    return train_df
#lets test it
for x in validation_windows:
    train_x = get_training_window(
        pre_validation_df,
        validation_month,
        x
    )

    print(f"{x}-month training window:")
    print("Rows:", len(train_x))
    print("Start month:", train_x["CloseMonth"].min())
    print("End month:", train_x["CloseMonth"].max())
    print()

3-month training window:
Rows: 26904
Start month: 2025-12
End month: 2026-02

6-month training window:
Rows: 60127
Start month: 2025-09
End month: 2026-02

9-month training window:
Rows: 95393
Start month: 2025-06
End month: 2026-02

10-month training window:
Rows: 107169
Start month: 2025-05
End month: 2026-02



Now we see that the candidate training windows were created sucessfully and we are using march 2026 as the validation month. From the cell above we see that the 3-month window uses data from December 2025 through February 2026 and contains 26,904 rows. The 6-month window uses September 2025 through February 2026 and contains 60,127 rows. The 9-month window uses June 2025 through February 2026 and contains 95,393 rows. The 10-month window uses all available data before the validation month, from May 2025 through February 2026, and contains 107,169 rows.

WSo we see that as the training window length increased the number of avaiable training observations also increased. This does not mean that a larger training set is better because we still need to factor the data containing homes that have oldersales records which will not represent the current market conditions properly.

Since this is a notebook for data preprecessing we will not continue with model training, lets instead clea missing values and encode/impute where necessary to get a clean dataset.

In [23]:
#Lets keep the historical/test structure since we didn't select optimal x
#clean copies
clean_historical_df = historical_df.copy()
clean_test_df = test_df.copy()

# impute numeric missing values using historical-data medians
# this avoids allowing information from April 2026 to influence preprocessing

imputation_values = {}
for col in numeric_impute_cols:
    median_value = clean_historical_df[col].median()
    imputation_values[col] = median_value
    clean_historical_df[col] = clean_historical_df[col].fillna(median_value)
    clean_test_df[col] = clean_test_df[col].fillna(median_value)
print("Numeric imputation values:")
for col, value in imputation_values.items():
    print(f"{col}: {value}")

#lets fill the missing categorical fields with "Unknown" so that we don't accidentally use misleading data for 
#specific categories, we will later do encoding as this will help us make "Unknonw" it's own category

# fill missing categorical values
for col in categorical_fill_cols:
    clean_historical_df[col] = (
        clean_historical_df[col]
        .astype("string")
        .fillna("Unknown"))
    clean_test_df[col] = (clean_test_df[col].astype("string").fillna("Unknown"))
cols_to_check = numeric_impute_cols + categorical_fill_cols
print("Historical missing values:")
print(clean_historical_df[cols_to_check].isna().sum())
print("\nTest missing values:")
print(clean_test_df[cols_to_check].isna().sum())
#now lets remove the prohibited features (which we defined before)
clean_historical_df = clean_historical_df.drop(
    columns=[
        col for col in prohibited_features
        if col in clean_historical_df.columns])
clean_test_df = clean_test_df.drop(
    columns=[
        col for col in prohibited_features
        if col in clean_test_df.columns])

Numeric imputation values:
LotSizeSquareFeet: 7293.0
YearBuilt: 1976.0
LivingArea: 1816.0
BathroomsTotalInteger: 2.0
Historical missing values:
LotSizeSquareFeet        0
YearBuilt                0
LivingArea               0
BathroomsTotalInteger    0
City                     0
PostalCode               0
dtype: int64

Test missing values:
LotSizeSquareFeet        0
YearBuilt                0
LivingArea               0
BathroomsTotalInteger    0
City                     0
PostalCode               0
dtype: int64


Finally lets only retain features that would be available for arbitrary California single-family properties at prediciton times.

Listing-specific fields, agent information, contract information, listing IDs,
and other sale-dependent variables are excluded.
`ClosePrice` is retained only as the prediction target, while `CloseDate` and
`CloseMonth` are retained only for chronological splitting.


In [24]:
# features currently approved for model input
final_features = [
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "YearBuilt",
    "PostalCode",
    "City",
    "LotSizeMissing"
]

# columns needed for target and time-based splitting
keep_cols = final_features + [
    "ClosePrice",
    "CloseDate",
    "CloseMonth"
]

clean_historical_df = clean_historical_df[keep_cols].copy()
clean_test_df = clean_test_df[keep_cols].copy()

print("Historical cleaned shape:", clean_historical_df.shape)
print("Test cleaned shape:", clean_test_df.shape)

print("\nColumns retained:")
print(clean_historical_df.columns.tolist())

# lets quickly verify the fields that we labeled as prohibited are truly gone
for col in prohibited_features:
    print(
        col,
        "FOUND" if col in clean_historical_df.columns else "removed"
    )

Historical cleaned shape: (118731, 11)
Test cleaned shape: (12489, 11)

Columns retained:
['LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'LotSizeSquareFeet', 'YearBuilt', 'PostalCode', 'City', 'LotSizeMissing', 'ClosePrice', 'CloseDate', 'CloseMonth']
ListPrice removed
OriginalListPrice removed
DaysOnMarket removed
PurchaseContractDate removed
ListingContractDate removed


In [ ]:
# now that we have verified we removed prohibited fields, we want to encode the categorical features

#first check how many unique categories we have
print("Unique cities:", clean_historical_df["City"].nunique())
print("Unique postal codes:", clean_historical_df["PostalCode"].nunique())

# we see that the numbers we got (Unique cities: 972 & Unique postal codes: 1996) tells us to not one-hot
#encode directly (this would cause us to create 3000 extra columns which is unpractical)
#lets classify unique values as others instead, so others becomes its own category.

#min # needed to make a category
min_category_count = 50
# find cities that occur at least 50 times in historical data
common_cities = (
    clean_historical_df["City"]
    .value_counts()
)

common_cities = common_cities[
    common_cities >= min_category_count
].index

# group rare/unseen cities into "Other"
clean_historical_df["City"] = clean_historical_df["City"].where(clean_historical_df["City"].isin(common_cities),"Other")
clean_test_df["City"] = clean_test_df["City"].where(
    clean_test_df["City"].isin(common_cities),
    "Other"
)

#now for zip code
# find postal codes that occur at least 50 times in historical data
common_postal_codes = (
    clean_historical_df["PostalCode"]
    .value_counts()
)

common_postal_codes = common_postal_codes[
    common_postal_codes >= min_category_count
].index

# group rare/unseen postal codes into "Other"
clean_historical_df["PostalCode"] = clean_historical_df["PostalCode"].where(clean_historical_df["PostalCode"].isin(common_postal_codes),"Other")

clean_test_df["PostalCode"] = clean_test_df["PostalCode"].where(
    clean_test_df["PostalCode"].isin(common_postal_codes),
    "Other"
)
print("Cities after grouping:", clean_historical_df["City"].nunique())
print("Postal codes after grouping:", clean_historical_df["PostalCode"].nunique())

# one-hot encode City and PostalCode
encoded_historical_df = pd.get_dummies(
    clean_historical_df,
    columns=["City", "PostalCode"],
    dtype=int)
encoded_test_df = pd.get_dummies(
    clean_test_df,
    columns=["City", "PostalCode"],
    dtype=int
)

# align historical and test datasets so they contain the same encoded columns
#we do this so we have matching columns which makes it easier to compare the datasets as they will have
#the same structure.
encoded_historical_df, encoded_test_df = encoded_historical_df.align(
    encoded_test_df,
    join="left",
    axis=1,
    fill_value=0
)
#this verifies if the two datasets hold same structure
print("Historical shape after encoding:", encoded_historical_df.shape)
print("Test shape after encoding:", encoded_test_df.shape)

print(
    "Same columns:",
    encoded_historical_df.columns.equals(encoded_test_df.columns)
)

Unique cities: 396
Unique postal codes: 681
Cities after grouping: 396
Postal codes after grouping: 681
Historical shape after encoding: (118731, 1086)
Test shape after encoding: (12489, 1086)
Same columns: True


Now we must keep in mind after encoding categoircal features and imputing numerical features that we must normalize Numeric features but its best to do that after the optimal trsaining window length is selected so that scaling parameters are learned only from the final slected training window to avoid data leakage.

Lets create the cleaned CSV file now

In [ ]:
# label the datasets before recombining
encoded_historical_df["DataSplit"] = "historical"
encoded_test_df["DataSplit"] = "test"
cleaned_df = pd.concat(
    [encoded_historical_df, encoded_test_df],
    ignore_index=True
)

print("Final cleaned shape:", cleaned_df.shape)

# final preprocessing checks
print("Final shape:", cleaned_df.shape)

print("\nMissing values remaining:")
print(cleaned_df.isna().sum().sum())

print("\nDataset split:")
print(cleaned_df["DataSplit"].value_counts())

print("\nClose date range:")
print(cleaned_df["CloseDate"].min())
print(cleaned_df["CloseDate"].max())

#letsd make sure the important columns are not missing before we export
important_cols = [
    "ClosePrice",
    "CloseDate",
    "CloseMonth",
    "DataSplit",
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "YearBuilt",
    "LotSizeMissing"
]

missing_important = [
    col for col in important_cols
    if col not in cleaned_df.columns
]

print("Important columns missing:")
print(missing_important)

#finally we export
output_path = os.path.join("Data", "cleaned_sfh_data.csv")

cleaned_df.to_csv(
    output_path,
    index=False
)

Final cleaned shape: (131220, 1087)
Final shape: (131220, 1087)

Missing values remaining:
0

Dataset split:
DataSplit
historical    118731
test           12489
Name: count, dtype: int64

Close date range:
2025-05-01 00:00:00
2026-04-30 00:00:00
Important columns missing:
[]
Cleaned dataset saved to: Data\cleaned_sfh_data.csv


## Preprocessing Summary

In this notebook we took the files from months may 2025 to april 2026 to combine and restrict them to California residential single-family properties.

Any features that depended on properties being listed or sold were excluded, this was done to ensure that there is no data leakage and that the model can be used for off-market properties. After we filled the numerical values using medians from historical data and assigned missing categorical values to a category labeled "Unknown". We then removed High cardinality values from the categorical variables and grouped unique values to a category labeled "other" so that we can use one-hot encoding.

We set data from April 2026 for testing set, earlier months were part of the historical pool, and the march 2026 data was set aside to be the validation set. We prepared the candidate training-window lengths for model comparison later.

Numerical features were not standardized in the exported dataset because the final
training-window length and model have not yet been selected. Scaling can be learned
from the selected training data later if required by the chosen model.

The final cleaned dataset contains no missing values and was exported as
`cleaned_sfh_data.csv`.